In [1]:
!pip install -q langchain langchain-community transformers sentence-transformers faiss-cpu pypdf huggingface_hub pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("llama_for_hydro")
login(token=secret_value_0)

In [3]:
import os
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from langchain.prompts import PromptTemplate

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [ ]:
pdf_dir = '/kaggle/input/hydrobot/'  
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith('.pdf')]

documents = []
for pdf in pdf_files:
    loader = PyMuPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} pages from {len(pdf_files)} PDFs.")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  
    chunk_overlap=200  
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks.")

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("faiss_index") 
print("Vector store created and saved.")

In [ ]:
model_name = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Auto-assign to GPU
    torch_dtype=torch.float16  # Half-precision for memory
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,  
    top_p=0.90,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=pipe)
print("Llama 2 loaded.")

In [ ]:
prompt_template = """Based on the following context about hydroponic farming, provide a clear, concise, and accurate answer to the question. Return only the answer, without including the context, instructions, or any other text.

Context: {context}

Question: {question}
Answer: """
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

#RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 8}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
import gradio as gr
def chatbot(query):
    result = qa_chain({"query": query})
    answer = result["result"]
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()
    return answer

# Gradio interface
iface = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="Hydroponics Education Chatbot",
    description="Ask questions about hydroponic farming!"
)

iface.launch(share=True) 